# Legal Clause Retriever

**Project 13** — Core RAG Projects

Build a retrieval system that finds relevant legal clauses from contracts,
highlights risky or missing clause patterns, and explains findings in plain language.

**Key Skills:** Domain-specific RAG, clause extraction, risk pattern matching,
gap analysis, legal NLP concepts.

> **Important Disclaimer:** This notebook is for **educational purposes only**.
> It is NOT a substitute for professional legal advice. Never rely on automated
> tools for real legal decisions without consulting a qualified attorney.

## Project Overview

Contracts contain dozens of clauses — liability limits, termination rights, IP
ownership, indemnification, confidentiality, and more. Reviewing them manually is
slow, expensive, and error-prone.

This notebook builds a **Legal Clause Retriever** that:
1. **Ingests** multiple contract documents with structured clause extraction
2. **Indexes** clauses for semantic search
3. **Retrieves** relevant clauses given a natural-language query
4. **Detects risky clause patterns** using rule-based heuristics
5. **Identifies missing clauses** by checking against a standard clause checklist
6. **Explains findings** in plain, non-legal language

### Architecture
```
Contracts → Clause Parser → Clause Embeddings → Vector Index
                                                      ↓
Query → Retrieval → Risk Analysis → Gap Analysis → Report
```

## Learning Goals

By the end of this notebook you will understand:
- How to parse semi-structured legal text into individual clauses
- How semantic retrieval works for domain-specific documents
- How to build rule-based risk detectors for legal clauses
- How gap analysis identifies missing protections in a contract
- The limitations of automated legal analysis (critical awareness)
- How to present legal findings responsibly with appropriate caveats

## Problem Statement

**Scenario:** A small business is reviewing three vendor contracts before signing.
They need to quickly:
- Find all clauses related to liability, termination, or data handling
- Identify potentially risky language (unlimited liability, auto-renewal traps,
  broad IP assignment)
- Check whether essential protections are missing (data breach notification,
  limitation of liability, dispute resolution)

**Goal:** Build a tool that makes contract review faster and highlights areas
that need legal attention — while being transparent about what it can and cannot do.

## Why This Project Matters

| Challenge | How Clause Retrieval Helps |
|-----------|--------------------------|
| Contracts are long and dense | Semantic search finds relevant clauses instantly |
| Legal language is hard to parse | System translates findings to plain English |
| Risky clauses hide in boilerplate | Pattern matching flags known risk patterns |
| Missing clauses go unnoticed | Gap analysis checks against a standard checklist |
| Manual review is expensive | Automated first-pass reduces attorney review time |

### Real-world context
Legal tech companies like Kira Systems, Luminance, and ContractPodAI use similar
approaches. This notebook teaches the core techniques behind these tools.

## ⚠️ Important Limitations and Ethical Considerations

**This tool is NOT legal advice.** It is a learning exercise that demonstrates
NLP techniques applied to legal text. Critical limitations:

1. **No legal training** — The system does not understand legal doctrine, case law,
   jurisdiction-specific rules, or how clauses interact with each other.

2. **Pattern matching ≠ legal analysis** — Flagging "unlimited liability" as risky
   is a keyword heuristic, not a legal opinion. Context matters enormously.

3. **False positives and negatives** — The system will miss some risks and flag some
   non-issues. It cannot replace a lawyer's judgment.

4. **Jurisdiction blindness** — Legal meaning depends on jurisdiction (US, UK, EU).
   This system does not account for jurisdictional differences.

5. **No precedent awareness** — The system doesn't know how courts have interpreted
   similar clauses in the past.

**Responsible use:** Use as a first-pass screening tool to prioritize areas for
human review, never as a final decision-maker.

## Environment Setup

Dependencies:
- **sentence-transformers** — for clause embeddings
- **chromadb** — for vector storage
- **numpy**, **sklearn** — for TF-IDF fallback and analysis

In [1]:
import os
import re
import json
import hashlib
import textwrap
import warnings
from collections import defaultdict, Counter

import numpy as np
warnings.filterwarnings("ignore")

# ── Optional dependencies ──
USE_CHROMA = False
USE_ST = False

try:
    import chromadb
    USE_CHROMA = True
    print("[OK] chromadb available")
except ImportError:
    print("[INFO] chromadb not available — using TF-IDF fallback")

try:
    from sentence_transformers import SentenceTransformer
    USE_ST = True
    print("[OK] sentence-transformers available")
except ImportError:
    print("[INFO] sentence-transformers not available — using TF-IDF fallback")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("\nSetup complete.")

[OK] chromadb available


[OK] sentence-transformers available

Setup complete.


In [2]:
# ── Configuration ──
TOP_K = 5
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))

print(f"Top-K: {TOP_K}, Embedding: {EMBEDDING_MODEL}")

Top-K: 5, Embedding: all-MiniLM-L6-v2


## Sample Contracts

We create three realistic sample contracts representing common business agreements.
Each contract is structured with numbered clauses covering standard topics.

These are **simplified educational examples** — real contracts are much longer and
more complex. The goal is to demonstrate the retrieval and analysis techniques.

In [3]:
# ── Sample contracts ──

CONTRACTS = [
    {
        "contract_id": "SaaS-2024-001",
        "title": "SaaS Platform Services Agreement",
        "contract_type": "SaaS Subscription",
        "parties": ["Acme Corp (Customer)", "CloudTech Inc (Provider)"],
        "effective_date": "2024-01-15",
        "clauses": [
            {
                "clause_id": "1.1",
                "heading": "Service Description",
                "category": "scope",
                "text": "Provider shall deliver the CloudTech Platform as a cloud-based software service, including data analytics dashboards, user management, and API access as described in Schedule A. The service shall maintain 99.9% uptime measured monthly, excluding scheduled maintenance windows communicated 48 hours in advance."
            },
            {
                "clause_id": "1.2",
                "heading": "Term and Renewal",
                "category": "term",
                "text": "This Agreement shall commence on the Effective Date and continue for an initial term of 36 months. The Agreement shall automatically renew for successive 12-month periods unless either party provides written notice of non-renewal at least 90 days prior to the end of the then-current term. Early termination during any term requires payment of 50% of remaining fees."
            },
            {
                "clause_id": "1.3",
                "heading": "Fees and Payment",
                "category": "payment",
                "text": "Customer shall pay the fees set forth in Schedule B. All fees are non-refundable. Provider reserves the right to increase fees by up to 15% upon each renewal term with 60 days written notice. Late payments shall accrue interest at 1.5% per month or the maximum rate permitted by law, whichever is lower."
            },
            {
                "clause_id": "1.4",
                "heading": "Data Ownership and Processing",
                "category": "data",
                "text": "Customer retains all ownership rights to Customer Data uploaded to the Platform. Provider may use aggregated, anonymized data derived from Customer Data for product improvement and benchmarking purposes. Provider shall process personal data in accordance with the Data Processing Addendum attached as Schedule C."
            },
            {
                "clause_id": "1.5",
                "heading": "Limitation of Liability",
                "category": "liability",
                "text": "IN NO EVENT SHALL PROVIDER'S TOTAL AGGREGATE LIABILITY EXCEED THE FEES PAID BY CUSTOMER IN THE TWELVE (12) MONTHS PRECEDING THE CLAIM. NEITHER PARTY SHALL BE LIABLE FOR ANY INDIRECT, INCIDENTAL, SPECIAL, CONSEQUENTIAL, OR PUNITIVE DAMAGES, INCLUDING LOST PROFITS OR LOST DATA, REGARDLESS OF THE CAUSE OF ACTION."
            },
            {
                "clause_id": "1.6",
                "heading": "Indemnification",
                "category": "indemnification",
                "text": "Provider shall indemnify and hold harmless Customer against any third-party claims arising from (a) infringement of intellectual property rights by the Platform, or (b) Provider's gross negligence or willful misconduct. Customer shall indemnify Provider against claims arising from Customer's use of the Platform in violation of this Agreement or applicable law."
            },
            {
                "clause_id": "1.7",
                "heading": "Confidentiality",
                "category": "confidentiality",
                "text": "Each party agrees to maintain the confidentiality of the other party's Confidential Information for a period of 5 years following disclosure. Confidential Information excludes information that is publicly available, independently developed, or rightfully received from a third party without restriction."
            },
            {
                "clause_id": "1.8",
                "heading": "Termination for Cause",
                "category": "termination",
                "text": "Either party may terminate this Agreement immediately upon written notice if the other party: (a) materially breaches this Agreement and fails to cure within 30 days of written notice, (b) becomes insolvent or files for bankruptcy, or (c) ceases to conduct business in the normal course. Upon termination, Provider shall make Customer Data available for export for 60 days."
            },
            {
                "clause_id": "1.9",
                "heading": "Governing Law",
                "category": "governing_law",
                "text": "This Agreement shall be governed by and construed in accordance with the laws of the State of Delaware, without regard to its conflict of laws provisions. Any disputes shall be resolved through binding arbitration in Wilmington, Delaware under the rules of the American Arbitration Association."
            },
            {
                "clause_id": "1.10",
                "heading": "Intellectual Property",
                "category": "ip",
                "text": "All intellectual property rights in the Platform, including any modifications, improvements, or derivative works, shall remain the exclusive property of Provider. Customer is granted a non-exclusive, non-transferable license to use the Platform during the term of this Agreement solely for Customer's internal business purposes."
            },
        ]
    },
    {
        "contract_id": "CONS-2024-002",
        "title": "Professional Services Consulting Agreement",
        "contract_type": "Consulting",
        "parties": ["DataDriven LLC (Client)", "TechConsult Partners (Consultant)"],
        "effective_date": "2024-03-01",
        "clauses": [
            {
                "clause_id": "2.1",
                "heading": "Scope of Services",
                "category": "scope",
                "text": "Consultant shall provide data engineering and machine learning consulting services as described in each Statement of Work (SOW). Services include system architecture design, model development, deployment support, and knowledge transfer sessions. Any services outside the SOW require written change order approval."
            },
            {
                "clause_id": "2.2",
                "heading": "Compensation",
                "category": "payment",
                "text": "Client shall pay Consultant at the rate of $250 per hour for senior consultants and $175 per hour for associate consultants. Invoices shall be submitted bi-weekly and paid within 30 days. Travel expenses shall be reimbursed at cost with receipts. No overtime premium applies."
            },
            {
                "clause_id": "2.3",
                "heading": "Intellectual Property Assignment",
                "category": "ip",
                "text": "All work product, deliverables, inventions, and intellectual property created by Consultant in the performance of services under this Agreement shall be the sole and exclusive property of Client, including all patent, copyright, and trade secret rights. Consultant hereby assigns all right, title, and interest in such work product to Client. Consultant retains rights to pre-existing tools and methodologies."
            },
            {
                "clause_id": "2.4",
                "heading": "Non-Solicitation",
                "category": "non_compete",
                "text": "During the term of this Agreement and for a period of 24 months following its termination, neither party shall directly or indirectly solicit, recruit, or hire any employee or contractor of the other party who was involved in the performance of services under this Agreement."
            },
            {
                "clause_id": "2.5",
                "heading": "Limitation of Liability",
                "category": "liability",
                "text": "Consultant's total liability under this Agreement shall not exceed the total fees paid by Client to Consultant in the six (6) months preceding the claim. Consultant shall not be liable for any business decisions made by Client based on Consultant's recommendations or deliverables."
            },
            {
                "clause_id": "2.6",
                "heading": "Confidentiality",
                "category": "confidentiality",
                "text": "Consultant shall maintain strict confidentiality of all Client data, business processes, trade secrets, and proprietary information. This obligation survives termination of this Agreement indefinitely. Consultant shall return or destroy all confidential materials within 10 days of termination."
            },
            {
                "clause_id": "2.7",
                "heading": "Term and Termination",
                "category": "termination",
                "text": "This Agreement shall continue until completion of all active SOWs or until terminated by either party. Either party may terminate with 30 days written notice. Client shall pay for all services rendered through the termination date. There is no early termination penalty."
            },
            {
                "clause_id": "2.8",
                "heading": "Warranty",
                "category": "warranty",
                "text": "Consultant warrants that services shall be performed in a professional and workmanlike manner consistent with industry standards. This warranty shall extend for 30 days following delivery of each deliverable. Consultant's sole remedy for breach of warranty is re-performance of the defective services."
            },
        ]
    },
    {
        "contract_id": "NDA-2024-003",
        "title": "Mutual Non-Disclosure Agreement",
        "contract_type": "NDA",
        "parties": ["InnovateTech Inc (Party A)", "StartupVentures (Party B)"],
        "effective_date": "2024-02-10",
        "clauses": [
            {
                "clause_id": "3.1",
                "heading": "Definition of Confidential Information",
                "category": "confidentiality",
                "text": "Confidential Information means any and all non-public information disclosed by either party, whether orally, in writing, or electronically, that is designated as confidential or that reasonably should be understood to be confidential given the nature of the information and circumstances of disclosure. This includes but is not limited to: source code, algorithms, customer lists, financial data, business plans, product roadmaps, and technical specifications."
            },
            {
                "clause_id": "3.2",
                "heading": "Obligations of Receiving Party",
                "category": "confidentiality",
                "text": "The Receiving Party shall: (a) use the Confidential Information solely for the Purpose, (b) protect it using at least the same degree of care used for its own confidential information, but no less than reasonable care, (c) limit disclosure to employees and contractors with a need to know who are bound by similar obligations, (d) not reverse engineer, decompile, or disassemble any Confidential Information."
            },
            {
                "clause_id": "3.3",
                "heading": "Exclusions",
                "category": "confidentiality",
                "text": "Confidential Information does not include information that: (a) was publicly known at the time of disclosure, (b) becomes publicly known through no fault of the Receiving Party, (c) was rightfully in the Receiving Party's possession before disclosure, (d) is independently developed without use of Confidential Information, (e) is rightfully received from a third party without restriction."
            },
            {
                "clause_id": "3.4",
                "heading": "Term and Duration",
                "category": "term",
                "text": "This Agreement shall be effective for a period of 2 years from the Effective Date. The confidentiality obligations shall survive termination and continue for 3 years following the expiration or termination of this Agreement."
            },
            {
                "clause_id": "3.5",
                "heading": "Return of Materials",
                "category": "termination",
                "text": "Upon termination or expiration, or upon request by the Disclosing Party, the Receiving Party shall promptly return or destroy all Confidential Information and any copies thereof. The Receiving Party shall certify in writing that all materials have been returned or destroyed, except for copies retained in automated backup systems which shall remain subject to the confidentiality obligations."
            },
            {
                "clause_id": "3.6",
                "heading": "Remedies",
                "category": "liability",
                "text": "Each party acknowledges that any breach of this Agreement may cause irreparable harm for which monetary damages would be inadequate. Accordingly, the non-breaching party shall be entitled to seek injunctive relief in addition to any other remedies available at law or in equity. The prevailing party in any legal action shall be entitled to recover reasonable attorney's fees."
            },
            {
                "clause_id": "3.7",
                "heading": "No License Granted",
                "category": "ip",
                "text": "Nothing in this Agreement grants any license or right to use the Disclosing Party's Confidential Information except as expressly set forth herein. All Confidential Information remains the exclusive property of the Disclosing Party. No patent, copyright, trademark, or other intellectual property rights are granted by implication or otherwise."
            },
        ]
    },
]

print(f"Created {len(CONTRACTS)} sample contracts:\n")
for c in CONTRACTS:
    print(f"  [{c['contract_id']}] {c['title']}")
    print(f"    Type: {c['contract_type']} | Parties: {', '.join(c['parties'])}")
    print(f"    Clauses: {len(c['clauses'])} | Date: {c['effective_date']}")
    print()

Created 3 sample contracts:

  [SaaS-2024-001] SaaS Platform Services Agreement
    Type: SaaS Subscription | Parties: Acme Corp (Customer), CloudTech Inc (Provider)
    Clauses: 10 | Date: 2024-01-15

  [CONS-2024-002] Professional Services Consulting Agreement
    Type: Consulting | Parties: DataDriven LLC (Client), TechConsult Partners (Consultant)
    Clauses: 8 | Date: 2024-03-01

  [NDA-2024-003] Mutual Non-Disclosure Agreement
    Type: NDA | Parties: InnovateTech Inc (Party A), StartupVentures (Party B)
    Clauses: 7 | Date: 2024-02-10



## Clause Parsing and Structuring

Each clause is extracted as a self-contained unit with rich metadata. The key
insight is that **clauses, not full documents**, are the atomic unit for legal
retrieval. A relevant answer almost always comes from one or two specific clauses,
not from reading the entire contract.

In [4]:
# ── Clause data model ──

class LegalClause:
    """Represents a single legal clause with full metadata."""

    def __init__(self, clause_dict, contract_meta):
        self.clause_id = clause_dict["clause_id"]
        self.heading = clause_dict["heading"]
        self.category = clause_dict["category"]
        self.text = clause_dict["text"]
        self.contract_id = contract_meta["contract_id"]
        self.contract_title = contract_meta["title"]
        self.contract_type = contract_meta["contract_type"]
        self.parties = contract_meta["parties"]
        self.effective_date = contract_meta["effective_date"]
        self.word_count = len(self.text.split())
        # Deterministic ID
        content_hash = hashlib.md5(
            f"{self.contract_id}:{self.clause_id}".encode()
        ).hexdigest()[:8]
        self.uid = f"{self.contract_id}-{self.clause_id}-{content_hash}"

    @property
    def metadata(self):
        return {
            "uid": self.uid,
            "clause_id": self.clause_id,
            "heading": self.heading,
            "category": self.category,
            "contract_id": self.contract_id,
            "contract_title": self.contract_title,
            "contract_type": self.contract_type,
            "effective_date": self.effective_date,
            "word_count": self.word_count,
        }

    @property
    def citation(self):
        return f"[{self.contract_id}] Clause {self.clause_id}: {self.heading}"

    def __repr__(self):
        return f"LegalClause({self.clause_id}, '{self.heading}', {self.word_count}w)"


# Parse all clauses from all contracts
all_clauses = []
for contract in CONTRACTS:
    meta = {
        "contract_id": contract["contract_id"],
        "title": contract["title"],
        "contract_type": contract["contract_type"],
        "parties": contract["parties"],
        "effective_date": contract["effective_date"],
    }
    for clause_dict in contract["clauses"]:
        clause = LegalClause(clause_dict, meta)
        all_clauses.append(clause)

print(f"Parsed {len(all_clauses)} clauses from {len(CONTRACTS)} contracts\n")
for clause in all_clauses:
    print(f"  {clause.citation} ({clause.word_count} words, category: {clause.category})")


Parsed 25 clauses from 3 contracts

  [SaaS-2024-001] Clause 1.1: Service Description (42 words, category: scope)
  [SaaS-2024-001] Clause 1.2: Term and Renewal (58 words, category: term)
  [SaaS-2024-001] Clause 1.3: Fees and Payment (53 words, category: payment)
  [SaaS-2024-001] Clause 1.4: Data Ownership and Processing (44 words, category: data)
  [SaaS-2024-001] Clause 1.5: Limitation of Liability (48 words, category: liability)
  [SaaS-2024-001] Clause 1.6: Indemnification (51 words, category: indemnification)
  [SaaS-2024-001] Clause 1.7: Confidentiality (40 words, category: confidentiality)
  [SaaS-2024-001] Clause 1.8: Termination for Cause (59 words, category: termination)
  [SaaS-2024-001] Clause 1.9: Governing Law (45 words, category: governing_law)
  [SaaS-2024-001] Clause 1.10: Intellectual Property (44 words, category: ip)
  [CONS-2024-002] Clause 2.1: Scope of Services (41 words, category: scope)
  [CONS-2024-002] Clause 2.2: Compensation (44 words, category: payment)
 

In [5]:
# ── Corpus statistics ──

categories = Counter(c.category for c in all_clauses)
contracts = Counter(c.contract_id for c in all_clauses)
word_counts = [c.word_count for c in all_clauses]

print("=== Legal Corpus Statistics ===")
print(f"  Total clauses: {len(all_clauses)}")
print(f"  Total words: {sum(word_counts):,}")
print(f"  Avg words/clause: {np.mean(word_counts):.0f}")
print(f"  Min/Max: {min(word_counts)} / {max(word_counts)} words")

print(f"\nClauses by category:")
for cat, count in categories.most_common():
    print(f"  {cat}: {count}")

print(f"\nClauses by contract:")
for cid, count in contracts.most_common():
    print(f"  {cid}: {count}")

=== Legal Corpus Statistics ===
  Total clauses: 25
  Total words: 1,216
  Avg words/clause: 49
  Min/Max: 34 / 64 words

Clauses by category:
  confidentiality: 5
  liability: 3
  termination: 3
  ip: 3
  scope: 2
  term: 2
  payment: 2
  data: 1
  indemnification: 1
  governing_law: 1
  non_compete: 1
  warranty: 1

Clauses by contract:
  SaaS-2024-001: 10
  CONS-2024-002: 8
  NDA-2024-003: 7


## Building the Clause Index

We embed each clause into a vector and store it in a searchable index.
Since legal clauses are self-contained units (unlike free-form text that needs
chunking), we embed each clause as-is — no additional chunking needed.

This is a key advantage of **structured legal text**: clauses are natural
retrieval units.

In [6]:
# ── Build clause vector index ──

class ClauseIndex:
    """Vector index for legal clause retrieval."""

    def __init__(self, clauses):
        self.clauses = clauses
        self.texts = [c.text for c in clauses]
        self.clause_map = {c.uid: c for c in clauses}

        if USE_ST and USE_CHROMA:
            self._build_chroma()
        else:
            self._build_tfidf()

    def _build_chroma(self):
        print("Building index with sentence-transformers + ChromaDB...")
        self.model = SentenceTransformer(EMBEDDING_MODEL)
        self.client = chromadb.Client()
        try:
            self.client.delete_collection("legal_clauses")
        except Exception:
            pass
        self.collection = self.client.create_collection(
            name="legal_clauses", metadata={"hnsw:space": "cosine"}
        )
        embeddings = self.model.encode(self.texts, show_progress_bar=False).tolist()
        ids = [c.uid for c in self.clauses]
        metadatas = [c.metadata for c in self.clauses]
        self.collection.add(ids=ids, embeddings=embeddings,
                            documents=self.texts, metadatas=metadatas)
        self.backend = "chroma"
        print(f"Indexed {self.collection.count()} clauses")

    def _build_tfidf(self):
        print("Building TF-IDF fallback index...")
        self.vectorizer = TfidfVectorizer(
            max_features=5000, stop_words="english", ngram_range=(1, 2)
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)
        self.backend = "tfidf"
        print(f"Indexed {self.tfidf_matrix.shape[0]} clauses ({self.tfidf_matrix.shape[1]} features)")

    def search(self, query, top_k=TOP_K, category_filter=None, contract_filter=None):
        """Search for relevant clauses with optional filters."""
        if self.backend == "chroma":
            return self._search_chroma(query, top_k, category_filter, contract_filter)
        return self._search_tfidf(query, top_k, category_filter, contract_filter)

    def _search_chroma(self, query, top_k, category_filter, contract_filter):
        where = None
        conditions = []
        if category_filter:
            conditions.append({"category": category_filter})
        if contract_filter:
            conditions.append({"contract_id": contract_filter})
        if len(conditions) == 1:
            where = conditions[0]
        elif len(conditions) > 1:
            where = {"$and": conditions}

        query_emb = self.model.encode([query]).tolist()
        results = self.collection.query(
            query_embeddings=query_emb,
            n_results=min(top_k, self.collection.count()),
            where=where,
        )
        output = []
        for i, uid in enumerate(results["ids"][0]):
            clause = self.clause_map[uid]
            dist = results["distances"][0][i]
            sim = 1.0 - dist
            output.append((clause, sim))
        return output

    def _search_tfidf(self, query, top_k, category_filter, contract_filter):
        candidates = []
        for i, c in enumerate(self.clauses):
            if category_filter and c.category != category_filter:
                continue
            if contract_filter and c.contract_id != contract_filter:
                continue
            candidates.append(i)
        if not candidates:
            return []
        query_vec = self.vectorizer.transform([query])
        cand_matrix = self.tfidf_matrix[candidates]
        sims = cosine_similarity(query_vec, cand_matrix).flatten()
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.clauses[candidates[i]], float(sims[i])) for i in top_idx]


clause_index = ClauseIndex(all_clauses)

Building index with sentence-transformers + ChromaDB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 25 clauses


## Baseline: Keyword Search

As always, we establish a keyword baseline before evaluating semantic retrieval.

In [7]:
# ── Baseline keyword search ──

def keyword_search(query, clauses, top_k=TOP_K):
    q_terms = set(query.lower().split())
    scored = []
    for clause in clauses:
        text_lower = clause.text.lower()
        hits = sum(1 for t in q_terms if t in text_lower)
        score = hits / max(len(q_terms), 1)
        scored.append((clause, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

test_q = "What is the limitation of liability?"
baseline = keyword_search(test_q, all_clauses)
print(f"Query: {test_q}\n")
print("--- Baseline (keyword) ---")
for clause, score in baseline[:3]:
    print(f"  Score: {score:.2f} | {clause.citation}")
    print(f"  Preview: {clause.text[:100]}...")
    print()

Query: What is the limitation of liability?

--- Baseline (keyword) ---
  Score: 0.50 | [SaaS-2024-001] Clause 1.2: Term and Renewal
  Preview: This Agreement shall commence on the Effective Date and continue for an initial term of 36 months. T...

  Score: 0.50 | [SaaS-2024-001] Clause 1.6: Indemnification
  Preview: Provider shall indemnify and hold harmless Customer against any third-party claims arising from (a) ...

  Score: 0.50 | [SaaS-2024-001] Clause 1.7: Confidentiality
  Preview: Each party agrees to maintain the confidentiality of the other party's Confidential Information for ...



## Semantic Retrieval

Testing the vector-based search across different legal topics.

In [8]:
# ── Semantic retrieval ──

test_q = "What is the limitation of liability?"
results = clause_index.search(test_q, top_k=3)

print(f"Query: {test_q}")
print(f"Backend: {clause_index.backend}\n")
for clause, score in results:
    print(f"  Score: {score:.3f} | {clause.citation}")
    print(f"  Contract: {clause.contract_title}")
    print(f"  Category: {clause.category}")
    print(f"  Text: {clause.text[:150]}...")
    print()

Query: What is the limitation of liability?
Backend: chroma

  Score: 0.600 | [SaaS-2024-001] Clause 1.5: Limitation of Liability
  Contract: SaaS Platform Services Agreement
  Category: liability
  Text: IN NO EVENT SHALL PROVIDER'S TOTAL AGGREGATE LIABILITY EXCEED THE FEES PAID BY CUSTOMER IN THE TWELVE (12) MONTHS PRECEDING THE CLAIM. NEITHER PARTY S...

  Score: 0.509 | [CONS-2024-002] Clause 2.5: Limitation of Liability
  Contract: Professional Services Consulting Agreement
  Category: liability
  Text: Consultant's total liability under this Agreement shall not exceed the total fees paid by Client to Consultant in the six (6) months preceding the cla...

  Score: 0.419 | [SaaS-2024-001] Clause 1.6: Indemnification
  Contract: SaaS Platform Services Agreement
  Category: indemnification
  Text: Provider shall indemnify and hold harmless Customer against any third-party claims arising from (a) infringement of intellectual property rights by th...



In [9]:
# ── Multi-topic retrieval ──

legal_queries = [
    "How can I terminate this contract?",
    "Who owns the intellectual property?",
    "What data protection measures are in place?",
    "What happens if there is a breach of confidentiality?",
    "What are the payment terms?",
    "Is there a non-compete or non-solicitation clause?",
]

print("=== Multi-Topic Legal Retrieval ===\n")
for q in legal_queries:
    results = clause_index.search(q, top_k=1)
    if results:
        clause, score = results[0]
        print(f"Q: {q}")
        print(f"  -> [{score:.3f}] {clause.citation} (category: {clause.category})")
        print(f"     {clause.text[:80]}...\n")

=== Multi-Topic Legal Retrieval ===

Q: How can I terminate this contract?
  -> [0.521] [CONS-2024-002] Clause 2.4: Non-Solicitation (category: non_compete)
     During the term of this Agreement and for a period of 24 months following its te...

Q: Who owns the intellectual property?
  -> [0.511] [CONS-2024-002] Clause 2.3: Intellectual Property Assignment (category: ip)
     All work product, deliverables, inventions, and intellectual property created by...

Q: What data protection measures are in place?
  -> [0.453] [SaaS-2024-001] Clause 1.4: Data Ownership and Processing (category: data)
     Customer retains all ownership rights to Customer Data uploaded to the Platform....

Q: What happens if there is a breach of confidentiality?
  -> [0.615] [SaaS-2024-001] Clause 1.7: Confidentiality (category: confidentiality)
     Each party agrees to maintain the confidentiality of the other party's Confident...

Q: What are the payment terms?
  -> [0.426] [SaaS-2024-001] Clause 1.2: Term a

In [10]:
# ── Filtered retrieval: search within one contract ──

print("=== Filtered: SaaS Agreement only ===\n")
q = "What happens to my data when the contract ends?"
results = clause_index.search(q, top_k=3, contract_filter="SaaS-2024-001")

for clause, score in results:
    print(f"  [{score:.3f}] {clause.citation}")
    print(f"  {clause.text[:120]}...\n")

print("=== Filtered: Confidentiality clauses only ===\n")
q = "How long must confidential information be protected?"
results = clause_index.search(q, top_k=3, category_filter="confidentiality")

for clause, score in results:
    print(f"  [{score:.3f}] {clause.citation}")
    print(f"  Contract: {clause.contract_title}")
    print(f"  {clause.text[:120]}...\n")

=== Filtered: SaaS Agreement only ===

  [0.502] [SaaS-2024-001] Clause 1.8: Termination for Cause
  Either party may terminate this Agreement immediately upon written notice if the other party: (a) materially breaches th...

  [0.421] [SaaS-2024-001] Clause 1.4: Data Ownership and Processing
  Customer retains all ownership rights to Customer Data uploaded to the Platform. Provider may use aggregated, anonymized...

  [0.341] [SaaS-2024-001] Clause 1.5: Limitation of Liability
  IN NO EVENT SHALL PROVIDER'S TOTAL AGGREGATE LIABILITY EXCEED THE FEES PAID BY CUSTOMER IN THE TWELVE (12) MONTHS PRECED...

=== Filtered: Confidentiality clauses only ===

  [0.821] [SaaS-2024-001] Clause 1.7: Confidentiality
  Contract: SaaS Platform Services Agreement
  Each party agrees to maintain the confidentiality of the other party's Confidential Information for a period of 5 years ...

  [0.620] [NDA-2024-003] Clause 3.1: Definition of Confidential Information
  Contract: Mutual Non-Disclosure Agreem

## Risk Pattern Detection

We define rule-based heuristics that flag potentially risky clause patterns.
Each risk pattern has:
- A **name** and **description**
- **Detection rules** (keyword patterns, structural checks)
- A **severity level** (HIGH, MEDIUM, LOW)
- An **explanation** in plain English

> **Caveat:** These are educational heuristics, not legal opinions.
> Whether a clause is actually "risky" depends heavily on context,
> negotiating position, jurisdiction, and the specific deal.

In [11]:
# ── Risk pattern definitions ──

RISK_PATTERNS = [
    {
        "name": "Unlimited Liability",
        "description": "No cap on liability exposure",
        "severity": "HIGH",
        "patterns": [
            r"unlimited\s+liabilit",
            r"no\s+limit\s+(on|to)\s+liabilit",
            r"full\s+and\s+complete\s+liabilit",
        ],
        "explanation": "If there is no liability cap, one party could be exposed to unlimited financial risk. Most contracts include a liability cap (e.g., fees paid in the last 12 months)."
    },
    {
        "name": "Auto-Renewal Trap",
        "description": "Contract auto-renews with short or no opt-out window",
        "severity": "MEDIUM",
        "patterns": [
            r"automatically\s+renew",
            r"auto[\s-]?renew",
        ],
        "check_fn_name": "check_auto_renewal",
        "explanation": "Auto-renewal is common but can be risky if the opt-out notice period is very long (90+ days) or if renewal locks in price increases."
    },
    {
        "name": "Broad IP Assignment",
        "description": "Overly broad intellectual property transfer",
        "severity": "HIGH",
        "patterns": [
            r"all\s+(?:right|title|interest)",
            r"sole\s+and\s+exclusive\s+property",
            r"hereby\s+assigns?\s+all",
        ],
        "explanation": "Broad IP assignment means one party gives up all rights to work created under the agreement. This may include pre-existing IP or future unrelated work if not properly scoped."
    },
    {
        "name": "Non-Refundable Fees",
        "description": "Fees cannot be recovered even if service is not delivered",
        "severity": "MEDIUM",
        "patterns": [
            r"non[\s-]?refundable",
            r"no\s+refund",
            r"fees\s+(?:are|shall\s+be)\s+non[\s-]?refundable",
        ],
        "explanation": "Non-refundable fees mean the customer cannot recover payments even if the provider fails to deliver. Consider adding service-level credits."
    },
    {
        "name": "Short Cure Period",
        "description": "Very short time to fix a breach before termination",
        "severity": "LOW",
        "patterns": [
            r"cure\s+within\s+(?:[1-9]|1[0-4])\s+days",
            r"(?:5|7|10)\s+(?:business\s+)?days?\s+(?:to|of)\s+(?:cure|remedy)",
        ],
        "explanation": "A short cure period (under 15 days) may not provide enough time to fix a complex issue before the other party can terminate."
    },
    {
        "name": "Indefinite Confidentiality",
        "description": "Confidentiality obligations that last forever",
        "severity": "MEDIUM",
        "patterns": [
            r"survive[s]?\s+(?:termination|expiration).*?indefinitely",
            r"perpetual\s+confidentiality",
            r"obligations?\s+(?:shall\s+)?survive.*?indefinitely",
        ],
        "explanation": "Indefinite confidentiality obligations can be burdensome. Most agreements limit confidentiality to 3-5 years after termination."
    },
    {
        "name": "Unilateral Price Increase",
        "description": "Provider can increase prices without consent",
        "severity": "MEDIUM",
        "patterns": [
            r"right\s+to\s+increase\s+fees",
            r"increase\s+(?:prices?|fees)\s+(?:by\s+)?up\s+to",
            r"adjust\s+(?:prices?|fees)\s+at\s+(?:its|their)\s+(?:sole\s+)?discretion",
        ],
        "explanation": "Unilateral price increases can lead to unexpected cost growth. Look for caps (e.g., max 5% per year) or opt-out rights on increase."
    },
    {
        "name": "Long Non-Solicitation",
        "description": "Non-solicitation period exceeds 12 months",
        "severity": "LOW",
        "patterns": [
            r"(?:18|24|36)\s+months?\s+(?:following|after)",
            r"(?:2|3)\s+years?\s+(?:following|after)\s+(?:its\s+)?termination",
        ],
        "explanation": "Non-solicitation periods over 12 months can restrict your ability to hire talent. Courts in some jurisdictions may not enforce overly long periods."
    },
]

print(f"Defined {len(RISK_PATTERNS)} risk patterns:")
for rp in RISK_PATTERNS:
    print(f"  [{rp['severity']:6s}] {rp['name']}: {rp['description']}")

Defined 8 risk patterns:
  [HIGH  ] Unlimited Liability: No cap on liability exposure
  [MEDIUM] Auto-Renewal Trap: Contract auto-renews with short or no opt-out window
  [HIGH  ] Broad IP Assignment: Overly broad intellectual property transfer
  [MEDIUM] Non-Refundable Fees: Fees cannot be recovered even if service is not delivered
  [LOW   ] Short Cure Period: Very short time to fix a breach before termination
  [MEDIUM] Indefinite Confidentiality: Confidentiality obligations that last forever
  [MEDIUM] Unilateral Price Increase: Provider can increase prices without consent
  [LOW   ] Long Non-Solicitation: Non-solicitation period exceeds 12 months


In [12]:
# ── Run risk detection across all clauses ──

def detect_risks(clause, risk_patterns):
    """Check a clause against all risk patterns. Returns list of matched risks."""
    matches = []
    text = clause.text
    for pattern in risk_patterns:
        for regex in pattern["patterns"]:
            if re.search(regex, text, re.IGNORECASE):
                matches.append({
                    "risk_name": pattern["name"],
                    "severity": pattern["severity"],
                    "clause": clause.citation,
                    "contract": clause.contract_title,
                    "category": clause.category,
                    "explanation": pattern["explanation"],
                    "matched_pattern": regex,
                })
                break  # One match per pattern per clause is enough
    return matches


# Scan all clauses
all_risks = []
for clause in all_clauses:
    clause_risks = detect_risks(clause, RISK_PATTERNS)
    all_risks.extend(clause_risks)

print(f"=== Risk Scan Results ===")
print(f"Total risks detected: {len(all_risks)}\n")

# Group by severity
by_severity = defaultdict(list)
for risk in all_risks:
    by_severity[risk["severity"]].append(risk)

for sev in ["HIGH", "MEDIUM", "LOW"]:
    risks = by_severity.get(sev, [])
    if risks:
        print(f"--- {sev} Severity ({len(risks)}) ---")
        for r in risks:
            print(f"  [{r['risk_name']}] {r['clause']}")
            print(f"    Contract: {r['contract']}")
            print(f"    Why: {r['explanation'][:100]}...")
            print()

=== Risk Scan Results ===
Total risks detected: 6

--- HIGH Severity (1) ---
  [Broad IP Assignment] [CONS-2024-002] Clause 2.3: Intellectual Property Assignment
    Contract: Professional Services Consulting Agreement
    Why: Broad IP assignment means one party gives up all rights to work created under the agreement. This ma...

--- MEDIUM Severity (4) ---
  [Auto-Renewal Trap] [SaaS-2024-001] Clause 1.2: Term and Renewal
    Contract: SaaS Platform Services Agreement
    Why: Auto-renewal is common but can be risky if the opt-out notice period is very long (90+ days) or if r...

  [Non-Refundable Fees] [SaaS-2024-001] Clause 1.3: Fees and Payment
    Contract: SaaS Platform Services Agreement
    Why: Non-refundable fees mean the customer cannot recover payments even if the provider fails to deliver....

  [Unilateral Price Increase] [SaaS-2024-001] Clause 1.3: Fees and Payment
    Contract: SaaS Platform Services Agreement
    Why: Unilateral price increases can lead to unexpected 

## Gap Analysis: Missing Clause Detection

Beyond finding risky clauses, it's important to check whether **essential clauses
are missing**. We define a checklist of clause categories that a well-drafted
contract should include, then check each contract against it.

Different contract types need different clauses — an NDA doesn't need payment
terms, and a SaaS agreement should have an SLA.

In [13]:
# ── Standard clause checklists by contract type ──

CLAUSE_CHECKLISTS = {
    "SaaS Subscription": {
        "required": [
            ("scope", "Service Description / SLA"),
            ("payment", "Fees and Payment Terms"),
            ("data", "Data Ownership / Data Processing"),
            ("liability", "Limitation of Liability"),
            ("termination", "Termination Rights"),
            ("confidentiality", "Confidentiality"),
            ("ip", "Intellectual Property"),
            ("governing_law", "Governing Law / Dispute Resolution"),
        ],
        "recommended": [
            ("indemnification", "Indemnification"),
            ("warranty", "Service Warranty / SLA Credits"),
            ("data_breach", "Data Breach Notification"),
            ("force_majeure", "Force Majeure"),
            ("insurance", "Insurance Requirements"),
        ]
    },
    "Consulting": {
        "required": [
            ("scope", "Scope of Services"),
            ("payment", "Compensation and Payment"),
            ("ip", "IP Ownership / Work Product"),
            ("confidentiality", "Confidentiality"),
            ("termination", "Termination Rights"),
            ("liability", "Limitation of Liability"),
        ],
        "recommended": [
            ("warranty", "Warranty of Services"),
            ("non_compete", "Non-Compete / Non-Solicitation"),
            ("indemnification", "Indemnification"),
            ("insurance", "Professional Liability Insurance"),
            ("governing_law", "Governing Law"),
        ]
    },
    "NDA": {
        "required": [
            ("confidentiality", "Confidentiality Definition and Obligations"),
            ("term", "Term and Duration"),
            ("termination", "Return/Destruction of Materials"),
            ("ip", "No License / IP Rights"),
        ],
        "recommended": [
            ("liability", "Remedies / Liability"),
            ("governing_law", "Governing Law"),
            ("exclusions", "Exclusions from Confidentiality"),
        ]
    },
}


def gap_analysis(contract, clauses, checklists):
    """Check which required and recommended clauses are present or missing."""
    contract_type = contract["contract_type"]
    checklist = checklists.get(contract_type, {})

    if not checklist:
        return {"contract_id": contract["contract_id"], "status": "no_checklist"}

    clause_categories = set(c.category for c in clauses if c.contract_id == contract["contract_id"])

    result = {
        "contract_id": contract["contract_id"],
        "contract_title": contract["title"],
        "contract_type": contract_type,
        "required_present": [],
        "required_missing": [],
        "recommended_present": [],
        "recommended_missing": [],
    }

    for cat, label in checklist.get("required", []):
        if cat in clause_categories:
            result["required_present"].append((cat, label))
        else:
            result["required_missing"].append((cat, label))

    for cat, label in checklist.get("recommended", []):
        if cat in clause_categories:
            result["recommended_present"].append((cat, label))
        else:
            result["recommended_missing"].append((cat, label))

    return result


# Run gap analysis
print("=== Gap Analysis ===\n")
for contract in CONTRACTS:
    gap = gap_analysis(contract, all_clauses, CLAUSE_CHECKLISTS)

    req_total = len(gap["required_present"]) + len(gap["required_missing"])
    req_pct = len(gap["required_present"]) / max(req_total, 1)
    rec_total = len(gap["recommended_present"]) + len(gap["recommended_missing"])
    rec_pct = len(gap["recommended_present"]) / max(rec_total, 1)

    print(f"[{gap['contract_id']}] {gap['contract_title']}")
    print(f"  Required clauses:    {len(gap['required_present'])}/{req_total} ({req_pct:.0%})")
    print(f"  Recommended clauses: {len(gap['recommended_present'])}/{rec_total} ({rec_pct:.0%})")

    if gap["required_missing"]:
        print(f"  MISSING (required):")
        for cat, label in gap["required_missing"]:
            print(f"    - {label} ({cat})")

    if gap["recommended_missing"]:
        print(f"  MISSING (recommended):")
        for cat, label in gap["recommended_missing"]:
            print(f"    - {label} ({cat})")
    print()

=== Gap Analysis ===

[SaaS-2024-001] SaaS Platform Services Agreement
  Required clauses:    8/8 (100%)
  Recommended clauses: 1/5 (20%)
  MISSING (recommended):
    - Service Warranty / SLA Credits (warranty)
    - Data Breach Notification (data_breach)
    - Force Majeure (force_majeure)
    - Insurance Requirements (insurance)

[CONS-2024-002] Professional Services Consulting Agreement
  Required clauses:    6/6 (100%)
  Recommended clauses: 2/5 (40%)
  MISSING (recommended):
    - Indemnification (indemnification)
    - Professional Liability Insurance (insurance)
    - Governing Law (governing_law)

[NDA-2024-003] Mutual Non-Disclosure Agreement
  Required clauses:    4/4 (100%)
  Recommended clauses: 1/3 (33%)
  MISSING (recommended):
    - Governing Law (governing_law)
    - Exclusions from Confidentiality (exclusions)



## Full Legal Clause Retriever Pipeline

Combining search, risk detection, and gap analysis into a unified query interface.

In [14]:
# ── Full Legal Clause Retriever ──

class LegalClauseRetriever:
    """Retrieves legal clauses with risk and gap analysis."""

    def __init__(self, clause_index, all_clauses, contracts, risk_patterns, checklists):
        self.index = clause_index
        self.all_clauses = all_clauses
        self.contracts = contracts
        self.risk_patterns = risk_patterns
        self.checklists = checklists

    def query(self, question, top_k=TOP_K, contract_filter=None,
              category_filter=None, show_risks=True):
        """Answer a legal query with retrieval, risk flags, and citations.

        Args:
            question: Natural language question about the contracts
            top_k: Number of clauses to retrieve
            contract_filter: Restrict to a specific contract ID
            category_filter: Restrict to a specific clause category
            show_risks: Whether to display risk analysis

        Returns:
            Dict with answer, clauses, risks, confidence
        """
        # Retrieve
        results = self.index.search(
            question, top_k=top_k,
            category_filter=category_filter,
            contract_filter=contract_filter,
        )

        if not results:
            return {"answer": "No relevant clauses found.", "clauses": [],
                    "risks": [], "confidence": 0.0}

        # Synthesize answer
        answer = self._synthesize(question, results)

        # Risk check on retrieved clauses
        risks = []
        if show_risks:
            for clause, _ in results[:3]:
                risks.extend(detect_risks(clause, self.risk_patterns))

        # Confidence
        scores = [s for _, s in results]
        confidence = self._confidence(scores)

        # Display
        self._display(question, answer, results, risks, confidence, contract_filter, category_filter)

        return {
            "answer": answer,
            "clauses": [{"citation": c.citation, "score": round(s, 3)} for c, s in results[:3]],
            "risks": risks,
            "confidence": confidence,
        }

    def _synthesize(self, question, results):
        """Build an answer from retrieved clauses."""
        q_terms = set(question.lower().split())
        parts = []
        for clause, score in results[:3]:
            sentences = re.split(r'(?<=[.!?])\s+', clause.text)
            relevant = []
            for sent in sentences:
                overlap = sum(1 for t in q_terms if t in sent.lower())
                if overlap > 0 and len(sent) > 20:
                    relevant.append((sent, overlap))
            relevant.sort(key=lambda x: x[1], reverse=True)
            best = [s for s, _ in relevant[:2]]
            if best:
                parts.append(f"{' '.join(best)} [{clause.clause_id}]")
        if parts:
            return "\n\n".join(parts)
        return f"{results[0][0].text[:250]}... [{results[0][0].clause_id}]"

    def _confidence(self, scores):
        top = scores[0]
        avg = np.mean(scores) if scores else 0
        conf = 0.6 * min(top / 0.7, 1.0) + 0.4 * min(avg / 0.5, 1.0)
        return round(min(conf, 1.0), 3)

    def _display(self, question, answer, results, risks, confidence, contract_filter, category_filter):
        print("=" * 70)
        print(f"QUERY: {question}")
        if contract_filter:
            print(f"  Contract filter: {contract_filter}")
        if category_filter:
            print(f"  Category filter: {category_filter}")
        print("=" * 70)

        print(f"\nANSWER:")
        for line in answer.split("\n"):
            if line.strip():
                wrapped = textwrap.fill(line.strip(), width=66,
                                        initial_indent="  ", subsequent_indent="  ")
                print(wrapped)

        level = "HIGH" if confidence >= 0.7 else ("MEDIUM" if confidence >= 0.4 else "LOW")
        bar = "#" * int(confidence * 25)
        print(f"\nCONFIDENCE: {confidence:.3f} ({level}) |{bar}|")

        print(f"\nSOURCE CLAUSES:")
        for i, (clause, score) in enumerate(results[:3], 1):
            print(f"  [{i}] {clause.citation} (score: {score:.3f})")
            print(f"      Contract: {clause.contract_title}")
            print(f"      Category: {clause.category}")

        if risks:
            print(f"\nRISK FLAGS:")
            for r in risks:
                print(f"  [{r['severity']}] {r['risk_name']}")
                print(f"    Clause: {r['clause']}")
                wrapped = textwrap.fill(r["explanation"], width=64,
                                        initial_indent="    Why: ", subsequent_indent="         ")
                print(wrapped)
        else:
            print(f"\n  No risk patterns detected in retrieved clauses.")

        print("=" * 70)


# Build the retriever
retriever = LegalClauseRetriever(
    clause_index, all_clauses, CONTRACTS, RISK_PATTERNS, CLAUSE_CHECKLISTS
)
print("Legal Clause Retriever ready.")

Legal Clause Retriever ready.


## Query Examples

In [15]:
# ── Query 1: Liability ──
retriever.query("What is the limitation of liability in each contract?")

QUERY: What is the limitation of liability in each contract?

ANSWER:
  IN NO EVENT SHALL PROVIDER'S TOTAL AGGREGATE LIABILITY EXCEED
  THE FEES PAID BY CUSTOMER IN THE TWELVE (12) MONTHS PRECEDING
  THE CLAIM. NEITHER PARTY SHALL BE LIABLE FOR ANY INDIRECT,
  INCIDENTAL, SPECIAL, CONSEQUENTIAL, OR PUNITIVE DAMAGES,
  INCLUDING LOST PROFITS OR LOST DATA, REGARDLESS OF THE CAUSE OF
  ACTION. [1.5]
  Consultant's total liability under this Agreement shall not
  exceed the total fees paid by Client to Consultant in the six
  (6) months preceding the claim. Consultant shall not be liable
  for any business decisions made by Client based on Consultant's
  recommendations or deliverables. [2.5]
  Each party acknowledges that any breach of this Agreement may
  cause irreparable harm for which monetary damages would be
  inadequate. Accordingly, the non-breaching party shall be
  entitled to seek injunctive relief in addition to any other
  remedies available at law or in equity. [3.6]

CONFID

{'answer': "IN NO EVENT SHALL PROVIDER'S TOTAL AGGREGATE LIABILITY EXCEED THE FEES PAID BY CUSTOMER IN THE TWELVE (12) MONTHS PRECEDING THE CLAIM. NEITHER PARTY SHALL BE LIABLE FOR ANY INDIRECT, INCIDENTAL, SPECIAL, CONSEQUENTIAL, OR PUNITIVE DAMAGES, INCLUDING LOST PROFITS OR LOST DATA, REGARDLESS OF THE CAUSE OF ACTION. [1.5]\n\nConsultant's total liability under this Agreement shall not exceed the total fees paid by Client to Consultant in the six (6) months preceding the claim. Consultant shall not be liable for any business decisions made by Client based on Consultant's recommendations or deliverables. [2.5]\n\nEach party acknowledges that any breach of this Agreement may cause irreparable harm for which monetary damages would be inadequate. Accordingly, the non-breaching party shall be entitled to seek injunctive relief in addition to any other remedies available at law or in equity. [3.6]",
 'clauses': [{'citation': '[SaaS-2024-001] Clause 1.5: Limitation of Liability',
   'scor

In [16]:
# ── Query 2: Termination ──
retriever.query("How can I terminate the contract and what happens to my data?")

QUERY: How can I terminate the contract and what happens to my data?

ANSWER:
  Either party may terminate this Agreement immediately upon
  written notice if the other party: (a) materially breaches this
  Agreement and fails to cure within 30 days of written notice,
  (b) becomes insolvent or files for bankruptcy, or (c) ceases to
  conduct business in the normal course. Upon termination,
  Provider shall make Customer Data available for export for 60
  days. [1.8]
  Consultant shall maintain strict confidentiality of all Client
  data, business processes, trade secrets, and proprietary
  information. This obligation survives termination of this
  Agreement indefinitely. [2.6]
  Upon termination or expiration, or upon request by the
  Disclosing Party, the Receiving Party shall promptly return or
  destroy all Confidential Information and any copies thereof. The
  Receiving Party shall certify in writing that all materials have
  been returned or destroyed, except for copies retained

{'answer': 'Either party may terminate this Agreement immediately upon written notice if the other party: (a) materially breaches this Agreement and fails to cure within 30 days of written notice, (b) becomes insolvent or files for bankruptcy, or (c) ceases to conduct business in the normal course. Upon termination, Provider shall make Customer Data available for export for 60 days. [1.8]\n\nConsultant shall maintain strict confidentiality of all Client data, business processes, trade secrets, and proprietary information. This obligation survives termination of this Agreement indefinitely. [2.6]\n\nUpon termination or expiration, or upon request by the Disclosing Party, the Receiving Party shall promptly return or destroy all Confidential Information and any copies thereof. The Receiving Party shall certify in writing that all materials have been returned or destroyed, except for copies retained in automated backup systems which shall remain subject to the confidentiality obligations. 

In [17]:
# ── Query 3: IP ownership ──
retriever.query("Who owns the intellectual property created during the engagement?")

QUERY: Who owns the intellectual property created during the engagement?

ANSWER:
  All work product, deliverables, inventions, and intellectual
  property created by Consultant in the performance of services
  under this Agreement shall be the sole and exclusive property of
  Client, including all patent, copyright, and trade secret
  rights. [2.3]
  All intellectual property rights in the Platform, including any
  modifications, improvements, or derivative works, shall remain
  the exclusive property of Provider. Customer is granted a non-
  exclusive, non-transferable license to use the Platform during
  the term of this Agreement solely for Customer's internal
  business purposes. [1.10]
  No patent, copyright, trademark, or other intellectual property
  rights are granted by implication or otherwise. All Confidential
  Information remains the exclusive property of the Disclosing
  Party. [3.7]

CONFIDENCE: 0.713 (HIGH) |#################|

SOURCE CLAUSES:
  [1] [CONS-2024-002] Cla

{'answer': "All work product, deliverables, inventions, and intellectual property created by Consultant in the performance of services under this Agreement shall be the sole and exclusive property of Client, including all patent, copyright, and trade secret rights. [2.3]\n\nAll intellectual property rights in the Platform, including any modifications, improvements, or derivative works, shall remain the exclusive property of Provider. Customer is granted a non-exclusive, non-transferable license to use the Platform during the term of this Agreement solely for Customer's internal business purposes. [1.10]\n\nNo patent, copyright, trademark, or other intellectual property rights are granted by implication or otherwise. All Confidential Information remains the exclusive property of the Disclosing Party. [3.7]",
 'clauses': [{'citation': '[CONS-2024-002] Clause 2.3: Intellectual Property Assignment',
   'score': 0.485},
  {'citation': '[SaaS-2024-001] Clause 1.10: Intellectual Property',
  

In [18]:
# ── Query 4: Filtered to one contract ──
retriever.query(
    "What are the payment terms and can prices increase?",
    contract_filter="SaaS-2024-001"
)

QUERY: What are the payment terms and can prices increase?
  Contract filter: SaaS-2024-001

ANSWER:
  Late payments shall accrue interest at 1.5% per month or the
  maximum rate permitted by law, whichever is lower. Customer
  shall pay the fees set forth in Schedule B. [1.3]
  This Agreement shall commence on the Effective Date and continue
  for an initial term of 36 months. The Agreement shall
  automatically renew for successive 12-month periods unless
  either party provides written notice of non-renewal at least 90
  days prior to the end of the then-current term. [1.2]
  IN NO EVENT SHALL PROVIDER'S TOTAL AGGREGATE LIABILITY EXCEED
  THE FEES PAID BY CUSTOMER IN THE TWELVE (12) MONTHS PRECEDING
  THE CLAIM. NEITHER PARTY SHALL BE LIABLE FOR ANY INDIRECT,
  INCIDENTAL, SPECIAL, CONSEQUENTIAL, OR PUNITIVE DAMAGES,
  INCLUDING LOST PROFITS OR LOST DATA, REGARDLESS OF THE CAUSE OF
  ACTION. [1.5]

CONFIDENCE: 0.726 (HIGH) |##################|

SOURCE CLAUSES:
  [1] [SaaS-2024-001] 

{'answer': "Late payments shall accrue interest at 1.5% per month or the maximum rate permitted by law, whichever is lower. Customer shall pay the fees set forth in Schedule B. [1.3]\n\nThis Agreement shall commence on the Effective Date and continue for an initial term of 36 months. The Agreement shall automatically renew for successive 12-month periods unless either party provides written notice of non-renewal at least 90 days prior to the end of the then-current term. [1.2]\n\nIN NO EVENT SHALL PROVIDER'S TOTAL AGGREGATE LIABILITY EXCEED THE FEES PAID BY CUSTOMER IN THE TWELVE (12) MONTHS PRECEDING THE CLAIM. NEITHER PARTY SHALL BE LIABLE FOR ANY INDIRECT, INCIDENTAL, SPECIAL, CONSEQUENTIAL, OR PUNITIVE DAMAGES, INCLUDING LOST PROFITS OR LOST DATA, REGARDLESS OF THE CAUSE OF ACTION. [1.5]",
 'clauses': [{'citation': '[SaaS-2024-001] Clause 1.3: Fees and Payment',
   'score': 0.533},
  {'citation': '[SaaS-2024-001] Clause 1.2: Term and Renewal', 'score': 0.356},
  {'citation': '[SaaS

In [19]:
# ── Query 5: Confidentiality across contracts ──
retriever.query(
    "How long do confidentiality obligations last?",
    category_filter="confidentiality"
)

QUERY: How long do confidentiality obligations last?
  Category filter: confidentiality

ANSWER:
  Each party agrees to maintain the confidentiality of the other
  party's Confidential Information for a period of 5 years
  following disclosure. [1.7]
  Consultant shall maintain strict confidentiality of all Client
  data, business processes, trade secrets, and proprietary
  information. [2.6]

CONFIDENCE: 1.000 (HIGH) |#########################|

SOURCE CLAUSES:
  [1] [SaaS-2024-001] Clause 1.7: Confidentiality (score: 0.777)
      Contract: SaaS Platform Services Agreement
      Category: confidentiality
  [2] [CONS-2024-002] Clause 2.6: Confidentiality (score: 0.601)
      Contract: Professional Services Consulting Agreement
      Category: confidentiality
  [3] [NDA-2024-003] Clause 3.1: Definition of Confidential Information (score: 0.488)
      Contract: Mutual Non-Disclosure Agreement
      Category: confidentiality

RISK FLAGS:
  [MEDIUM] Indefinite Confidentiality
    Clause: [

{'answer': "Each party agrees to maintain the confidentiality of the other party's Confidential Information for a period of 5 years following disclosure. [1.7]\n\nConsultant shall maintain strict confidentiality of all Client data, business processes, trade secrets, and proprietary information. [2.6]",
 'clauses': [{'citation': '[SaaS-2024-001] Clause 1.7: Confidentiality',
   'score': 0.777},
  {'citation': '[CONS-2024-002] Clause 2.6: Confidentiality', 'score': 0.601},
  {'citation': '[NDA-2024-003] Clause 3.1: Definition of Confidential Information',
   'score': 0.488}],
 'risks': [{'risk_name': 'Indefinite Confidentiality',
   'severity': 'MEDIUM',
   'clause': '[CONS-2024-002] Clause 2.6: Confidentiality',
   'contract': 'Professional Services Consulting Agreement',
   'category': 'confidentiality',
   'explanation': 'Indefinite confidentiality obligations can be burdensome. Most agreements limit confidentiality to 3-5 years after termination.',
   'matched_pattern': 'survive[s]?\

## Full Risk Report

Generating a comprehensive risk report across all contracts.

In [20]:
# ── Full risk report ──

print("=" * 70)
print("COMPREHENSIVE RISK REPORT")
print("=" * 70)

for contract in CONTRACTS:
    cid = contract["contract_id"]
    contract_clauses = [c for c in all_clauses if c.contract_id == cid]
    contract_risks = []
    for clause in contract_clauses:
        contract_risks.extend(detect_risks(clause, RISK_PATTERNS))

    print(f"\n--- [{cid}] {contract['title']} ---")
    print(f"Type: {contract['contract_type']} | Clauses: {len(contract_clauses)}")

    if contract_risks:
        high = [r for r in contract_risks if r["severity"] == "HIGH"]
        med = [r for r in contract_risks if r["severity"] == "MEDIUM"]
        low = [r for r in contract_risks if r["severity"] == "LOW"]
        print(f"Risks: {len(high)} HIGH, {len(med)} MEDIUM, {len(low)} LOW")
        for r in contract_risks:
            print(f"  [{r['severity']:6s}] {r['risk_name']}: {r['clause']}")
    else:
        print("  No risk patterns detected.")

    # Gap analysis
    gap = gap_analysis(contract, all_clauses, CLAUSE_CHECKLISTS)
    if gap.get("required_missing"):
        print(f"  Missing required clauses:")
        for cat, label in gap["required_missing"]:
            print(f"    - {label}")
    if gap.get("recommended_missing"):
        print(f"  Missing recommended clauses:")
        for cat, label in gap["recommended_missing"]:
            print(f"    - {label}")

print("\n" + "=" * 70)
print("Note: This is an automated first-pass analysis.")
print("All findings should be reviewed by a qualified attorney.")
print("=" * 70)

COMPREHENSIVE RISK REPORT

--- [SaaS-2024-001] SaaS Platform Services Agreement ---
Type: SaaS Subscription | Clauses: 10
Risks: 0 HIGH, 3 MEDIUM, 0 LOW
  [MEDIUM] Auto-Renewal Trap: [SaaS-2024-001] Clause 1.2: Term and Renewal
  [MEDIUM] Non-Refundable Fees: [SaaS-2024-001] Clause 1.3: Fees and Payment
  [MEDIUM] Unilateral Price Increase: [SaaS-2024-001] Clause 1.3: Fees and Payment
  Missing recommended clauses:
    - Service Warranty / SLA Credits
    - Data Breach Notification
    - Force Majeure
    - Insurance Requirements

--- [CONS-2024-002] Professional Services Consulting Agreement ---
Type: Consulting | Clauses: 8
Risks: 1 HIGH, 1 MEDIUM, 1 LOW
  [HIGH  ] Broad IP Assignment: [CONS-2024-002] Clause 2.3: Intellectual Property Assignment
  [LOW   ] Long Non-Solicitation: [CONS-2024-002] Clause 2.4: Non-Solicitation
  [MEDIUM] Indefinite Confidentiality: [CONS-2024-002] Clause 2.6: Confidentiality
  Missing recommended clauses:
    - Indemnification
    - Professional Liabilit

## Evaluation

We evaluate retrieval accuracy by checking whether the correct clause category
is returned for each test query.

In [21]:
# ── Evaluation ──

eval_set = [
    {"query": "What is the liability cap?",
     "expected_category": "liability"},
    {"query": "How do I terminate?",
     "expected_category": "termination"},
    {"query": "Who owns the IP?",
     "expected_category": "ip"},
    {"query": "What are the payment terms?",
     "expected_category": "payment"},
    {"query": "What is considered confidential?",
     "expected_category": "confidentiality"},
    {"query": "What services are included?",
     "expected_category": "scope"},
    {"query": "How long is the contract term?",
     "expected_category": "term"},
    {"query": "Is there a non-solicitation clause?",
     "expected_category": "non_compete"},
]

baseline_hits = 0
semantic_hits = 0

print(f"{'Query':<45} {'Baseline':^10} {'Semantic':^10}")
print("-" * 65)

for e in eval_set:
    q = e["query"]
    expected = e["expected_category"]

    b = keyword_search(q, all_clauses, top_k=1)
    b_hit = b[0][0].category == expected if b else False
    baseline_hits += int(b_hit)

    s = clause_index.search(q, top_k=1)
    s_hit = s[0][0].category == expected if s else False
    semantic_hits += int(s_hit)

    print(f"  {q:<43} {'HIT' if b_hit else 'MISS':^10} {'HIT' if s_hit else 'MISS':^10}")

n = len(eval_set)
print("-" * 65)
print(f"  {'ACCURACY':<43} {baseline_hits/n:^10.1%} {semantic_hits/n:^10.1%}")
print(f"\nBaseline: {baseline_hits}/{n}, Semantic: {semantic_hits}/{n}")

Query                                          Baseline   Semantic 
-----------------------------------------------------------------
  What is the liability cap?                     HIT        HIT    


  How do I terminate?                            MISS       HIT    
  Who owns the IP?                               MISS       HIT    
  What are the payment terms?                    HIT        MISS   


  What is considered confidential?               MISS       HIT    


  What services are included?                    HIT        HIT    
  How long is the contract term?                 MISS       HIT    


  Is there a non-solicitation clause?            MISS       HIT    
-----------------------------------------------------------------
  ACCURACY                                      37.5%      87.5%   

Baseline: 3/8, Semantic: 7/8


## Error Analysis

In [22]:
# ── Edge case analysis ──

edge_queries = [
    "Can I use the software after the contract ends?",  # Implied from term/license
    "What if the provider goes bankrupt?",                # Termination for cause
    "Are there any hidden fees?",                         # Vague
    "Is this contract fair?",                             # Subjective
]

print("=== Edge Cases ===\n")
for q in edge_queries:
    results = clause_index.search(q, top_k=2)
    top_score = results[0][1] if results else 0
    sources = [c.citation for c, _ in results[:2]]

    print(f"Q: \"{q}\"")
    print(f"  Top score: {top_score:.3f}")
    print(f"  Retrieved: {sources}")
    if top_score < 0.3:
        print(f"  -> Low confidence: query may be out of scope")
    print()

=== Edge Cases ===



Q: "Can I use the software after the contract ends?"
  Top score: 0.509
  Retrieved: ['[CONS-2024-002] Clause 2.7: Term and Termination', '[CONS-2024-002] Clause 2.4: Non-Solicitation']

Q: "What if the provider goes bankrupt?"
  Top score: 0.588
  Retrieved: ['[SaaS-2024-001] Clause 1.5: Limitation of Liability', '[SaaS-2024-001] Clause 1.6: Indemnification']



Q: "Are there any hidden fees?"
  Top score: 0.441
  Retrieved: ['[SaaS-2024-001] Clause 1.3: Fees and Payment', '[NDA-2024-003] Clause 3.7: No License Granted']



Q: "Is this contract fair?"
  Top score: 0.398
  Retrieved: ['[NDA-2024-003] Clause 3.6: Remedies', '[CONS-2024-002] Clause 2.4: Non-Solicitation']



In [23]:
# ── Export metrics ──

total_risks = len(all_risks)
high_risks = sum(1 for r in all_risks if r["severity"] == "HIGH")
med_risks = sum(1 for r in all_risks if r["severity"] == "MEDIUM")

metrics = {
    "project": "Legal Clause Retriever",
    "corpus": {
        "contracts": len(CONTRACTS),
        "total_clauses": len(all_clauses),
        "categories": list(set(c.category for c in all_clauses)),
    },
    "retrieval_backend": clause_index.backend,
    "evaluation": {
        "total_queries": len(eval_set),
        "baseline_accuracy": baseline_hits / len(eval_set),
        "semantic_accuracy": semantic_hits / len(eval_set),
    },
    "risk_analysis": {
        "patterns_defined": len(RISK_PATTERNS),
        "total_risks_found": total_risks,
        "high_severity": high_risks,
        "medium_severity": med_risks,
        "low_severity": total_risks - high_risks - med_risks,
    },
}

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\GenAI\13_Legal_Clause_Retriever\metrics.json
{
  "project": "Legal Clause Retriever",
  "corpus": {
    "contracts": 3,
    "total_clauses": 25,
    "categories": [
      "data",
      "liability",
      "scope",
      "indemnification",
      "confidentiality",
      "governing_law",
      "payment",
      "ip",
      "non_compete",
      "warranty",
      "termination",
      "term"
    ]
  },
  "retrieval_backend": "chroma",
  "evaluation": {
    "total_queries": 8,
    "baseline_accuracy": 0.375,
    "semantic_accuracy": 0.875
  },
  "risk_analysis": {
    "patterns_defined": 8,
    "total_risks_found": 6,
    "high_severity": 1,
    "medium_severity": 4,
    "low_severity": 1
  }
}


## Limitations

1. **Not legal advice** — Cannot replace attorney review. Automated analysis
   misses context, intent, and interplay between clauses.

2. **Rule-based risk detection** — Only catches patterns we explicitly define.
   New or unusual risk patterns will be missed entirely.

3. **No cross-clause reasoning** — The system analyzes clauses individually but
   doesn't understand how they interact (e.g., liability cap modified by
   an indemnification clause).

4. **No jurisdiction awareness** — Legal meaning varies by jurisdiction.
   An "unreasonable" non-compete in California is different from one in Texas.

5. **Simple gap analysis** — Checks for category presence only. A contract could
   have a "liability" clause that is actually inadequate, but the gap check
   would mark it as present.

6. **No negotiation context** — Whether a clause is "risky" depends on which
   side you're on. The system doesn't know your negotiating position.

7. **Small sample corpus** — Real legal RAG systems train on thousands of
   contracts and use fine-tuned legal language models.

## Common Mistakes

1. **Treating automated findings as legal conclusions** — A risk flag means
   "this pattern exists," not "this is legally problematic." Always have a
   lawyer review flagged clauses.

2. **Ignoring clause interactions** — A limitation of liability clause might
   be acceptable *because* there's a strong indemnification clause. Analyzing
   clauses in isolation misses these relationships.

3. **Over-relying on keyword matching** — Legal concepts can be expressed in
   many ways. "The Provider assumes no responsibility" and "Limitation of
   Liability" refer to the same concept but use different words.

4. **Not accounting for contract type** — A broad IP assignment is normal in
   a work-for-hire consulting agreement but alarming in a SaaS subscription.

5. **Forgetting about definitions** — Legal contracts often define terms in a
   definitions section. The word "Confidential Information" in clause 5 means
   whatever clause 1 defined it to mean.

6. **Presenting findings without caveats** — Always include disclaimers and
   encourage human review. Overconfident automated legal analysis is dangerous.

## Mini Challenge

### Exercise 1: Add a New Risk Pattern
Define a new risk pattern for "Force Majeure with broad exclusions" — when a
force majeure clause allows either party to suspend obligations for vague reasons.

### Exercise 2: Cross-Contract Comparison
Write code to compare the same clause category (e.g., "liability") across all
three contracts side by side. Which contract has the most favorable terms?

### Exercise 3: Plain English Summary
Modify `_synthesize()` to produce a simpler plain-English explanation instead
of returning clause text verbatim. Use templates like:
"This contract limits liability to [amount]. This means..."

### Exercise 4: Add a Contract
Create a fourth contract (e.g., an Employment Agreement) with clauses for
compensation, non-compete, termination, and severance. Run the full analysis.

### Exercise 5: Severity Scoring
Instead of binary risk detection, score each risk on a 1-10 scale based on
how many aggravating patterns appear in the same clause.

## Production Considerations

1. **Legal LLM** — Use a fine-tuned legal language model (LegalBERT, law-specific
   GPT-4 system prompt) for better clause understanding and answer generation.

2. **OCR Pipeline** — Real contracts come as PDFs. Add an OCR/PDF extraction
   layer (PyMuPDF, pdfplumber, or DocTR) before clause parsing.

3. **Clause Classification** — Train a classifier to automatically categorize
   clauses (liability, termination, IP, etc.) instead of relying on contract
   structure.

4. **Custom Risk Models** — Train risk classifiers on labeled data from legal
   reviewers rather than using keyword patterns.

5. **Audit Trail** — Log all queries, retrievals, and risk flags for compliance
   and accountability.

6. **Human-in-the-Loop** — Design the UI so flagged risks always route to a
   human reviewer. Never auto-approve or auto-reject based on automated analysis.

7. **Multi-language** — Support contracts in multiple languages using
   multilingual embeddings (XLM-RoBERTa) or translation preprocessing.

8. **Version Tracking** — Track clause changes across contract revisions to
   identify what was added, removed, or modified.

## How to Improve This Project

- **Clause-level embeddings** — Fine-tune embeddings on legal text
  (e.g., train on CUAD dataset) for better legal semantic matching.
- **Cross-clause reasoning** — Build a graph of clause relationships and
  analyze how liability, indemnification, and termination interact.
- **Negotiation assistant** — Given a flagged risk, suggest alternative
  language that would reduce the risk.
- **Benchmark scoring** — Compare each contract against industry standards
  and assign an overall "favorability score" per contract.
- **Template matching** — Compare clauses against standard templates
  (e.g., UNCITRAL, ICC, standard NDAs) to identify deviations.

## Key Takeaways

1. **Clauses are natural retrieval units** — unlike free-form text, legal
   contracts have structure. Use clauses as embedding units, not arbitrary chunks.

2. **Risk detection needs rules + retrieval** — keyword patterns catch known
   risks, but semantic retrieval surfaces them even when phrased differently.

3. **Gap analysis catches what risk detection misses** — a missing clause is
   just as important as a risky one. Checking against checklists is essential.

4. **Always present findings with caveats** — automated legal analysis must
   be framed as a first-pass screening tool, not as legal advice.

5. **Context is everything in legal NLP** — the same clause can be perfectly
   reasonable or extremely risky depending on context, jurisdiction, and
   negotiating position.

6. **Metadata filtering improves precision** — searching "liability clauses only"
   or "within the SaaS agreement" dramatically improves result quality.